# Task 2: MySQL Database Pipeline – TechReads Retail Analytics
**Module:** CMP-X304-0 Data Engineering  
**Database:** techreads_db  
**Table:** books  
**Input:** techreads_books.csv

## Step 1: Import Required Libraries

In [1]:
# mysql-connector-python: official MySQL driver for Python
# allows us to connect to MySQL and execute SQL commands from Python
import mysql.connector

# pandas: used to read the CSV file and inspect data before insertion
import pandas as pd

## Step 2: Load and Inspect CSV Data

In [2]:
# Load the CSV file produced by Task 1
df = pd.read_csv(r'd:\Trae\DECW1---TechReads-Retail-Analytics\Task 1\techreads_books.csv', 
                 encoding='utf-8-sig',
                 na_values=['N/A', 'NA', 'n/a', ''])

# Preview the data to confirm it loaded correctly
print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

Total rows: 20
Columns: ['Title', 'Author', 'Year', 'Rating', 'Price']

Data types:
Title         str
Author    float64
Year      float64
Rating      int64
Price     float64
dtype: object

First 5 rows:


,Title,Author,Year,Rating,Price
0,Algorithms to Live By: The Computer Science of...,NaN,NaN,1,30.81
1,The Mathews Men: Seven Brothers and the War Ag...,NaN,NaN,5,42.91
2,Tracing Numbers on a Train,NaN,NaN,3,41.60
3,Everydata: The Misinformation Hidden in the Li...,NaN,NaN,2,54.35
4,"At The Existentialist Café: Freedom, Being, an...",NaN,NaN,5,29.93


## Step 3: Connect to MySQL and Create Database

In [3]:
# Establish connection to MySQL server
# Update 'password' to match your local MySQL root password
conn = mysql.connector.connect(
    host='localhost',
    port=8868,
    user='root',
    password='123123'  
)
cursor = conn.cursor()

# Create the database if it does not already exist
cursor.execute('CREATE DATABASE IF NOT EXISTS techreads_db')
print('Database techreads_db created (or already exists).')

# Select the database for all subsequent operations
cursor.execute('USE techreads_db')
print('Using database: techreads_db')

Database techreads_db created (or already exists).
Using database: techreads_db


## Step 4: Create the Books Table

In [4]:
# Drop the table if it already exists to allow clean re-runs during testing
cursor.execute('DROP TABLE IF EXISTS books')

# Create the books table with appropriate data types and constraints
# - id: auto-incrementing primary key, uniquely identifies each record
# - title: VARCHAR(255) for variable-length book titles, NOT NULL as title is essential
# - price: DECIMAL(6,2) for accurate monetary values (up to 9999.99)
# - star_rating: TINYINT to store integer ratings 1-5, efficient for small numbers
# - availability: VARCHAR(50) for stock status text
# - category: VARCHAR(100) for book category classification
create_table_query = '''
CREATE TABLE books (
    id      INT AUTO_INCREMENT PRIMARY KEY,
    title   VARCHAR(255) NOT NULL,
    author  VARCHAR(255),
    year    VARCHAR(10),
    rating  TINYINT,
    price   DECIMAL(6,2)
)
''';
cursor.execute(create_table_query)
print('Table books created successfully.')

# Verify the table structure using DESCRIBE
cursor.execute('DESCRIBE books')
print('\nTable structure:')
for row in cursor.fetchall():
    print(row)

Table books created successfully.

Table structure:
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('title', 'varchar(255)', 'NO', '', None, '')
('author', 'varchar(255)', 'YES', '', None, '')
('year', 'varchar(10)', 'YES', '', None, '')
('rating', 'tinyint', 'YES', '', None, '')
('price', 'decimal(6,2)', 'YES', '', None, '')


## Step 5: Import CSV Data into MySQL

In [5]:
# SQL INSERT statement using parameterised queries
# %s placeholders prevent SQL injection and handle special characters in titles safely
insert_query = '''
    INSERT INTO books (title, author, year, rating, price)
    VALUES (%s, %s, %s, %s, %s)
''';

# Iterate through each row in the DataFrame and insert into MySQL
inserted = 0
for _, row in df.iterrows():
    # Convert NaN values to None for MySQL compatibility
    title = row['Title'] if not pd.isna(row['Title']) else None
    author = row['Author'] if not pd.isna(row['Author']) else None
    year = row['Year'] if not pd.isna(row['Year']) else None
    rating = int(row['Rating']) if not pd.isna(row['Rating']) else None
    price = float(row['Price']) if not pd.isna(row['Price']) else None
    
    cursor.execute(insert_query, (
        title,
        author,
        year,
        rating,
        price
    ))
    inserted += 1

# Commit the transaction to permanently save all changes to the database
conn.commit()
print(f'Successfully inserted {inserted} records into books table.')

Successfully inserted 20 records into books table.


## Step 6: SQL Queries

In [6]:
# Query 1: Select title, star_rating, price – sorted by price ascending
# Requirement: extract three columns and sort by one column (price)
query1 = '''
    SELECT title, rating, price
    FROM books
    ORDER BY price ASC
''';
cursor.execute(query1)
results1 = cursor.fetchall()

print('Query 1: All books ordered by price (low to high)')
print(f'{"Title":<65} {"Rating":<8} {"Price"}')
print('-' * 85)
for row in results1:
    print(f'{str(row[0]):<65} {str(row[1]):<8} £{row[2]}')

Query 1: All books ordered by price (low to high)
Title                                                             Rating   Price
-------------------------------------------------------------------------------------
Code Name Verity (Code Name Verity #1)                            4        £22.13
The End of Faith: Religion, Terror, and the Future of Reason      4        £22.13
The Mysterious Affair at Styles (Hercule Poirot #1)               4        £24.80
Talking to Girls About Duran Duran: One Young Man's Quest for True Love and a Cooler Haircut 4        £25.15
At The Existentialist Café: Freedom, Being, and apricot cocktails with: Jean-Paul Sartre, Simone de Beauvoir, Albert Camus, Martin Heidegger, Edmund Husserl, Karl Jaspers, Maurice Merleau-Ponty and others 5        £29.93
A Walk in the Woods: Rediscovering America on the Appalachian Trail 4        £30.48
Algorithms to Live By: The Computer Science of Human Decisions    1        £30.81
Twenty Love Poems and a Song of Despair  

In [7]:
# Query 2: Select title, price, author – filter books with high ratings
# Identifies well-rated books, useful for publisher recommendation insights
query2 = '''
    SELECT title, price, author
    FROM books
    WHERE rating >= 4
    ORDER BY rating DESC
''';
cursor.execute(query2)
results2 = cursor.fetchall()

print('Query 2: Books with star rating >= 4')
print(f'{"Title":<65} {"Price":<10} {"Author"}')
print('-' * 90)
for row in results2:
    print(f'{str(row[0]):<65} £{str(row[1]):<9} {row[2]}')

Query 2: Books with star rating >= 4
Title                                                             Price      Author
------------------------------------------------------------------------------------------
The Mathews Men: Seven Brothers and the War Against Hitler's U-boats £42.91     None
At The Existentialist Café: Freedom, Being, and apricot cocktails with: Jean-Paul Sartre, Simone de Beauvoir, Albert Camus, Martin Heidegger, Edmund Husserl, Karl Jaspers, Maurice Merleau-Ponty and others £29.93     None
Rip it Up and Start Again                                         £35.02     None
Done Rubbed Out (Reightman & Bailey #1)                           £37.72     None
Brain on Fire: My Month of Madness                                £49.32     None
Barefoot Contessa at Home: Everyday Recipes You'll Make Over and Over Again £50.62     None
A Piece of Sky, a Grain of Rice: A Memoir in Four Meditations     £56.76     None
Digital Fortress                                              

In [8]:
# Query 3: Select title, author, price – top 5 most expensive books
# Useful for pricing trend analysis across categories
query3 = '''
    SELECT title, author, price
    FROM books
    ORDER BY price
    DESC LIMIT 5
''';
cursor.execute(query3)
results3 = cursor.fetchall()

print('Query 3: Top 5 most expensive books')
print(f'{"Title":<65} {"Author":<20} {"Price"}')
print('-' * 95)
for row in results3:
    print(f'{str(row[0]):<65} {str(row[1]):<20} £{row[2]}')

Query 3: Top 5 most expensive books
Title                                                             Author               Price
-----------------------------------------------------------------------------------------------
Digital Fortress                                                  None                 £58.00
A Piece of Sky, a Grain of Rice: A Memoir in Four Meditations     None                 £56.76
Everydata: The Misinformation Hidden in the Little Data You Consume Every Day None                 £54.35
Barefoot Contessa at Home: Everyday Recipes You'll Make Over and Over Again None                 £50.62
Brain on Fire: My Month of Madness                                None                 £49.32


## Step 7: Close Connection

In [9]:
# Always close the cursor and connection when finished
# This releases database resources and prevents connection leaks
cursor.close()
conn.close()
print('MySQL connection closed.')

MySQL connection closed.
